# samples

> Offline fee-market specimens for fast local development.

`slashbtc` keeps live Bitcoin Core RPC on the edge of the workflow. The visualization layer works against compact `FeeBlock` objects, so notebooks, tests, and design iteration can run without a node.

In [ ]:
#| default_exp samples

In [ ]:
#| hide
from nbdev.showdoc import *

## Default path

Start with a deterministic synthetic sample when iterating on visual design or transformations.

In [ ]:
#| eval: false
from slashbtc.samples import sample_fee_block
from slashbtc.viz import fee_distribution_chart

block = sample_fee_block("synthetic_modern_busy")
fig = fee_distribution_chart(block)

## Synthetic samples

Synthetic samples are deterministic and cover useful fee-market shapes:

- `synthetic_modern_busy`: dense modern block with a visible fee tail.
- `synthetic_calm`: low-fee market with a small tail.
- `synthetic_congestion_tail`: high-fee stress case with a 200+ sat/vB overflow bucket.
- `synthetic_zero_fee_era`: block-100,000-style zero-fee edge case.
- `synthetic_empty`: coinbase-only edge case.

## Real specimens

Packaged real-mainnet fixtures are compact summaries derived from Bitcoin Core `getblock` verbosity `3` responses. They keep only the fields needed for fee distribution work: block identity, timestamp, transaction count, virtual size, and fee sats.

- `mainnet_zero_fee_100000`: confirms the early-chain zero-fee behavior.
- `mainnet_recent_947663`: recent mainnet block with enough fee variety for chart work.

## Refreshing fixtures

Use live RPC only when minting or refreshing a specimen.

In [ ]:
#| eval: false
# Reads BITCOIN_RPC_URL and BITCOIN_RPC_COOKIE from .env by default.
# You can also pass --url, --cookie, --rpc-user, and --rpc-password explicitly.
#
# slashbtc-capture-fee-block 947663 \
#   --out slashbtc/data/fee_blocks/mainnet_recent_947663.json.gz

In [ ]:
#| export
import math
import random
from dataclasses import dataclass
from importlib import resources
from pathlib import Path
from typing import Any

from slashbtc.fee import FeeBlock, FeeTx, capture_fee_block, load_fee_block, save_fee_block

FIXTURE_PACKAGE = "slashbtc"
FIXTURE_DIR = "data/fee_blocks"
DEFAULT_SAMPLE = "synthetic_modern_busy"


@dataclass(frozen=True)
class SyntheticFeeProfile:
    tx_count: int
    median_fee_sat_vb: float
    sigma: float
    seed: int
    height: int
    block_time: int
    zero_fee_share: float = 0.0
    tail_share: float = 0.05
    tail_multiplier: float = 6.0
    consolidation_share: float = 0.08


SYNTHETIC_PROFILES: dict[str, SyntheticFeeProfile] = {
    "synthetic_modern_busy": SyntheticFeeProfile(
        tx_count=2847,
        median_fee_sat_vb=18,
        sigma=0.62,
        seed=842913,
        height=842913,
        block_time=1747733650,
        tail_share=0.05,
        tail_multiplier=7.5,
    ),
    "synthetic_calm": SyntheticFeeProfile(
        tx_count=1850,
        median_fee_sat_vb=3,
        sigma=0.48,
        seed=700001,
        height=700001,
        block_time=1631333672,
        zero_fee_share=0.01,
        tail_share=0.03,
        tail_multiplier=4.0,
    ),
    "synthetic_congestion_tail": SyntheticFeeProfile(
        tx_count=3600,
        median_fee_sat_vb=72,
        sigma=0.85,
        seed=800008,
        height=800008,
        block_time=1690751730,
        tail_share=0.10,
        tail_multiplier=9.0,
    ),
    "synthetic_zero_fee_era": SyntheticFeeProfile(
        tx_count=3,
        median_fee_sat_vb=0,
        sigma=0.0,
        seed=100000,
        height=100000,
        block_time=1293623863,
        zero_fee_share=1.0,
        tail_share=0.0,
        tail_multiplier=1.0,
    ),
    "synthetic_empty": SyntheticFeeProfile(
        tx_count=0,
        median_fee_sat_vb=0,
        sigma=0.0,
        seed=10000,
        height=10000,
        block_time=1238988213,
        zero_fee_share=1.0,
        tail_share=0.0,
        tail_multiplier=1.0,
    ),
}


def sample_fee_block(name: str = DEFAULT_SAMPLE) -> FeeBlock:
    """Return a deterministic synthetic or packaged real fee block sample."""
    if name in SYNTHETIC_PROFILES:
        return synthetic_fee_block(name)
    return packaged_fee_block(name)


def sample_fee_block_names() -> list[str]:
    """List available synthetic and packaged fixture sample names."""
    names = set(SYNTHETIC_PROFILES)
    try:
        fixture_root = resources.files(FIXTURE_PACKAGE).joinpath(FIXTURE_DIR)
        for item in fixture_root.iterdir():
            if item.name.endswith(".json.gz"):
                names.add(item.name.removesuffix(".json.gz"))
            elif item.name.endswith(".json"):
                names.add(item.name.removesuffix(".json"))
    except FileNotFoundError:
        pass
    return sorted(names)


def synthetic_fee_block(name: str = DEFAULT_SAMPLE) -> FeeBlock:
    """Generate a deterministic block-shaped fee distribution by profile name."""
    try:
        profile = SYNTHETIC_PROFILES[name]
    except KeyError as exc:
        available = ", ".join(sorted(SYNTHETIC_PROFILES))
        raise KeyError(f"unknown synthetic fee profile {name!r}; available: {available}") from exc

    rng = random.Random(profile.seed)
    txs = tuple(_synthetic_txs(profile, rng))
    return FeeBlock(
        txs=txs,
        height=profile.height,
        block_time=profile.block_time,
        tx_count=len(txs) + 1,
        source=name,
    )


def packaged_fee_block(name: str) -> FeeBlock:
    """Load a compact real-mainnet fixture bundled with the package."""
    root = resources.files(FIXTURE_PACKAGE).joinpath(FIXTURE_DIR)
    candidates = [root.joinpath(f"{name}.json.gz"), root.joinpath(f"{name}.json")]
    for candidate in candidates:
        if candidate.is_file():
            with resources.as_file(candidate) as path:
                return load_fee_block(path)
    available = ", ".join(sample_fee_block_names())
    raise KeyError(f"unknown fee block sample {name!r}; available: {available}")


def capture_sample_fee_block(
    chain: Any,
    height_or_hash: int | str,
    name: str,
    out_dir: str | Path | None = None,
    compressed: bool = True,
) -> Path:
    """Fetch a live RPC block once and store it as a compact reusable sample."""
    out_root = Path(out_dir) if out_dir is not None else Path("slashbtc") / FIXTURE_DIR
    suffix = ".json.gz" if compressed else ".json"
    path = out_root / f"{name}{suffix}"
    block = capture_fee_block(chain, height_or_hash)
    return save_fee_block(block, path)


def _synthetic_txs(profile: SyntheticFeeProfile, rng: random.Random) -> list[FeeTx]:
    if profile.tx_count <= 0:
        return []

    txs: list[FeeTx] = []
    median = max(profile.median_fee_sat_vb, 0.0)
    log_median = math.log(max(median, 0.1))

    for _ in range(profile.tx_count):
        vsize = _synthetic_vsize(profile, rng)
        if rng.random() < profile.zero_fee_share or median == 0:
            fee_rate = 0.0
        else:
            fee_rate = rng.lognormvariate(log_median, profile.sigma)
            if rng.random() < profile.tail_share:
                fee_rate *= rng.uniform(profile.tail_multiplier * 0.5, profile.tail_multiplier * 1.5)
        fee_sats = int(round(fee_rate * vsize))
        if fee_rate > 0 and fee_sats == 0:
            fee_sats = 1
        txs.append(FeeTx(vsize_vb=vsize, fee_sats=max(0, fee_sats)))
    return txs


def _synthetic_vsize(profile: SyntheticFeeProfile, rng: random.Random) -> int:
    if rng.random() < profile.consolidation_share:
        return max(220, min(9500, int(rng.lognormvariate(math.log(2200), 0.7))))
    return max(110, min(1800, int(rng.lognormvariate(math.log(230), 0.55))))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()